In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error

# ==========================================
# Step 3.1: Load Data & Train Unconstrained Tree
# ==========================================
print("--- Step 3.1: Loading Data & Training Unconstrained Tree ---")
housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Unconstrained Decision Tree
dt_unconstrained = DecisionTreeRegressor(random_state=42)
dt_unconstrained.fit(X_train, y_train)

# Predictions & Evaluation
rmse_train_unc = root_mean_squared_error(y_train, dt_unconstrained.predict(X_train))
rmse_test_unc = root_mean_squared_error(y_test, dt_unconstrained.predict(X_test))

print(f"Unconstrained Tree - Train RMSE: {rmse_train_unc:.4f}")
print(f"Unconstrained Tree - Test RMSE:  {rmse_test_unc:.4f}\n")


# ==========================================
# Step 3.2: Implement 5-Fold Cross-Validation
# ==========================================
print("--- Step 3.2: 5-Fold Cross-Validation ---")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    dt_unconstrained, X_train, y_train, 
    scoring='neg_root_mean_squared_error', cv=kf
)
cv_rmse = -cv_scores

print(f"5-Fold CV RMSE per fold: {np.round(cv_rmse, 4)}")
print(f"Mean CV RMSE: {cv_rmse.mean():.4f} (+/- {cv_rmse.std():.4f})\n")


# ==========================================
# Step 3.3: Hyperparameter Tuning via GridSearchCV
# ==========================================
print("--- Step 3.3: GridSearchCV Tuning ---")
param_grid = {
    'max_depth': [4, 6, 8, 10, 12, 15, None],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_dt = grid_search.best_estimator_
print(f"Best Hyperparameters: {grid_search.best_params_}")

rmse_train_tuned = root_mean_squared_error(y_train, best_dt.predict(X_train))
rmse_test_tuned = root_mean_squared_error(y_test, best_dt.predict(X_test))

print(f"Tuned Tree - Train RMSE: {rmse_train_tuned:.4f}")
print(f"Tuned Tree - Test RMSE:  {rmse_test_tuned:.4f}\n")


# ==========================================
# Step 3.4: Train Baseline Models
# ==========================================
print("--- Step 3.4: Training Baseline Models ---")
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_train_rmse = root_mean_squared_error(y_train, lr.predict(X_train))
lr_test_rmse = root_mean_squared_error(y_test, lr.predict(X_test))

# Ridge Regression
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train, y_train)
ridge_train_rmse = root_mean_squared_error(y_train, ridge.predict(X_train))
ridge_test_rmse = root_mean_squared_error(y_test, ridge.predict(X_test))


# ==========================================
# Step 3.5: Final Model Comparison Report
# ==========================================
print("--- Step 3.5: Summary Comparison Report ---")
results = pd.DataFrame({
    'Model Name': ['Linear Regression', 'Ridge Regression', 'Unconstrained Tree', 'Tuned Decision Tree'],
    'Train RMSE': [lr_train_rmse, ridge_train_rmse, rmse_train_unc, rmse_train_tuned],
    'Test RMSE': [lr_test_rmse, ridge_test_rmse, rmse_test_unc, rmse_test_tuned],
    'Generalization Gap': [
        abs(lr_train_rmse - lr_test_rmse),
        abs(ridge_train_rmse - ridge_test_rmse),
        abs(rmse_train_unc - rmse_test_unc),
        abs(rmse_train_tuned - rmse_test_tuned)
    ]
})

print(results.to_string(index=False))

--- Step 3.1: Loading Data & Training Unconstrained Tree ---
Unconstrained Tree - Train RMSE: 0.0000
Unconstrained Tree - Test RMSE:  0.7037

--- Step 3.2: 5-Fold Cross-Validation ---
5-Fold CV RMSE per fold: [0.7574 0.6884 0.7358 0.7429 0.7422]
Mean CV RMSE: 0.7333 (+/- 0.0236)

--- Step 3.3: GridSearchCV Tuning ---
Best Hyperparameters: {'max_depth': 12, 'min_samples_leaf': 5, 'min_samples_split': 50}
Tuned Tree - Train RMSE: 0.4943
Tuned Tree - Test RMSE:  0.6040

--- Step 3.4: Training Baseline Models ---
--- Step 3.5: Summary Comparison Report ---
         Model Name   Train RMSE  Test RMSE  Generalization Gap
  Linear Regression 7.196757e-01   0.745581            0.025906
   Ridge Regression 7.196758e-01   0.745522            0.025847
 Unconstrained Tree 3.218326e-16   0.703729            0.703729
Tuned Decision Tree 4.942530e-01   0.604037            0.109784
